# 使用MSN数据聚类（Notebook 跑通版）

本 notebook 调用 `msn_pipeline` 包，完成：

加载MSN数据，图拓扑特征 → 使用GCN获取池化层特征 → 聚类 → 保存结果。

**使用方式**：从项目根目录或从本 `notebook/` 目录运行均可；若未执行过 `pip install -e .`，第一个代码单元会自动把 `src` 加入路径。

## 1. 环境与路径

In [ ]:
import sys
from pathlib import Path

# 自动识别项目根：当前目录或上一级（在 notebook/ 下运行时）
ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "msn_pipeline").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

print("项目根:", ROOT)
print("msn_pipeline 可导入:", (ROOT / "src" / "msn_pipeline").exists())

In [ ]:
import msn_pipeline
import msn_pipeline.pipelines   
from msn_pipeline.pipelines import run_clustering_with_topology_and_gcn
from msn_pipeline.pipelines import run_patient_clustering_with_stability

In [ ]:
pttopu_root = ROOT / "results" / "patient_subject_topology.csv"
gcn_feature_csv =  ROOT / "results" / "patient_gcn_features.csv"

## 2-1. Kmeans，n=3

In [ ]:

result = run_clustering_with_topology_and_gcn(
    patient_topology_csv=pttopu_root,
    gcn_feature_csv=gcn_feature_csv,
    method="kmeans",
    n_clusters=3,
    out_csv=ROOT / "results" / "cluster_result" / "kmeans3.csv",
)


subject_ids = result["subject_ids"]
features = result["features"]
labels = result["labels"]
print("聚类诊断:", result["diagnostics"])
print("标签 (前 10):", labels[:10])
print("输出文件:", result["output_csv"])

## 2-2.bootstrap ARI


In [ ]:


stable_result = run_patient_clustering_with_stability(
    patient_topology_csv=pttopu_root,
    gcn_feature_csv=gcn_feature_csv,
    method="kmeans",
    n_clusters=3,
    n_bootstrap=100,
    sample_ratio=0.8,
    out_csv=ROOT / "results" / "cluster_result" / "kmeans3-1.csv",
)



In [ ]:
s = stable_result["stability"]["summary"]
print(
    f"Bootstrap stability (pairwise ARI): "
    f"{s['mean_ari']:.3f} ± {s['std_ari']:.3f}, "
    f"95% CI [{s['ci95_low']:.3f}, {s['ci95_high']:.3f}]"
)
# 如需画图：
import matplotlib.pyplot as plt

ari_vals = stable_result["stability"]["ari_pairwise"]
plt.hist(ari_vals, bins=20)
plt.xlabel("Pairwise ARI")
plt.ylabel("Count")
plt.title("Clustering stability (bootstrap pairwise ARI)")
plt.show()

## 3. 多种聚类方法比较（固定 n_clusters）

使用 `strategies.py` 中的方法：**kmeans**、**ward**、**hierarchical**（complete）、**average**、**spectral**、**gmm**，在相同特征与 n_clusters=2，3 下跑一遍，用内部指标（轮廓系数、Davies-Bouldin、Calinski-Harabasz）比较。

In [ ]:
import pandas as pd

#methods = ["kmeans", "ward", "hierarchical", "average", "spectral", "gmm"]
methods = ["average"]
n_clusters_list = [2, 3]
rows = []

for n_clusters in n_clusters_list:
    for method in methods:
        r = run_clustering_with_topology_and_gcn(
            patient_topology_csv=pttopu_root,
            gcn_feature_csv=gcn_feature_csv,
            method=method,
            n_clusters=n_clusters,
            out_csv=ROOT / "results" / "cluster_result" / f"{method}_k{n_clusters}.csv",
        )
        diag = r["diagnostics"]
        row = {"method": method, "n_clusters": n_clusters}
        row["silhouette"] = diag.get("silhouette")
        row["davies_bouldin"] = diag.get("davies_bouldin")
        row["calinski_harabasz"] = diag.get("calinski_harabasz")
        if "inertia" in diag:
            row["inertia"] = diag["inertia"]
        if "bic" in diag:
            row["bic"] = diag["bic"]
            row["aic"] = diag["aic"]
        rows.append(row)

df_methods = pd.DataFrame(rows)
print("多种方法比较（n_clusters=2 与 3）：")
display(df_methods)

In [ ]:
# 从 df_methods 自动选出 Silhouette 最高、DB 最低的 (method, n_clusters)
best_sil = df_methods.loc[df_methods["silhouette"].idxmax()]
best_db = df_methods.loc[df_methods["davies_bouldin"].idxmin()]
rec_method = best_sil["method"]
rec_n = int(best_sil["n_clusters"])
print("推荐 (按 Silhouette 最高):", rec_method, "n_clusters =", rec_n)
print("  Silhouette =", round(best_sil["silhouette"], 4), ", Davies-Bouldin =", round(best_sil["davies_bouldin"], 4))
print("推荐 (按 Davies-Bouldin 最低):", best_db["method"], "n_clusters =", int(best_db["n_clusters"]))
print("  Silhouette =", round(best_db["silhouette"], 4), ", Davies-Bouldin =", round(best_db["davies_bouldin"], 4))
print()
print("建议: 方法 =", rec_method, ", n_clusters =", rec_n)
print("理由: 轮廓系数越高、Davies-Bouldin 越低，聚类结构越清晰；取 Silhouette 最优组合作为默认建议。")

**推荐解读**：上方的「建议」基于 `df_methods` 中 **Silhouette 最大**的 (method, n_clusters)。若 Silhouette 与 Davies-Bouldin 的推荐不一致，可结合业务先验（如希望 2 类还是 3 类）或对推荐方案跑一次 **bootstrap 稳定性**（`run_patient_clustering_with_stability`）再定。

In [ ]:
# 对average，n=2做 bootstrap ARI
stable_result = run_patient_clustering_with_stability(
    patient_topology_csv=pttopu_root,
    gcn_feature_csv=gcn_feature_csv,
    method="average",
    n_clusters=2,
    n_bootstrap=100,
    sample_ratio=0.8,
    out_csv=ROOT / "results" / "cluster_result" / "average2-1.csv",
)

s = stable_result["stability"]["summary"]
print(
    f"Bootstrap stability (pairwise ARI): "
    f"{s['mean_ari']:.3f} ± {s['std_ari']:.3f}, "
    f"95% CI [{s['ci95_low']:.3f}, {s['ci95_high']:.3f}]"
)
# 如需画图：
import matplotlib.pyplot as plt

ari_vals = stable_result["stability"]["ari_pairwise"]
plt.hist(ari_vals, bins=20)
plt.xlabel("Pairwise ARI")
plt.ylabel("Count")
plt.title("Clustering stability (bootstrap pairwise ARI)")
plt.show()



0.789 ± 0.309：不同 bootstrap 子集上得到的聚类结果之间，平均一致性是 0.789，标准差 0.309，说明有时很一致、有时波动较大。

95% CI [-0.030, 1.000]：
下界 -0.030 接近 0，表示在少数 bootstrap 中，两次聚类几乎像随机划分（ARI≈0），甚至略差。
上界 1.000 表示有时两次聚类完全一致。

## 4. 同一方法不同聚类数比较（如 KMeans k=2,3,4,5）

固定方法（如 kmeans），遍历不同 n_clusters，收集轮廓系数、Davies-Bouldin、Calinski-Harabasz（及 kmeans 的 inertia），用于选 k。

In [ ]:
import pandas as pd
# 固定方法，变化聚类数
method = "kmeans"
k_list = [2, 3, 4, 5, 6]
rows_k = []

for k in k_list:
    r = run_clustering_with_topology_and_gcn(
        patient_topology_csv=pttopu_root,
        gcn_feature_csv=gcn_feature_csv,
        method=method,
        n_clusters=k,
        out_csv=None,
    )
    diag = r["diagnostics"]
    row = {"method": method, "n_clusters": k}
    row["silhouette"] = diag.get("silhouette")
    row["davies_bouldin"] = diag.get("davies_bouldin")
    row["calinski_harabasz"] = diag.get("calinski_harabasz")
    if "inertia" in diag:
        row["inertia"] = diag["inertia"]
    rows_k.append(row)

df_k = pd.DataFrame(rows_k)
print(f"同一方法 {method} 不同 k 的内部指标：")
display(df_k)

In [ ]:
import matplotlib.pyplot as plt
# 可视化：轮廓系数、肘部（inertia）随 k 变化
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(df_k["n_clusters"], df_k["silhouette"], "o-")
axes[0].set_xlabel("n_clusters (k)")
axes[0].set_ylabel("Silhouette")
axes[0].set_title("Profile coefficient (the higher, the better)")
if "inertia" in df_k.columns:
    axes[1].plot(df_k["n_clusters"], df_k["inertia"], "o-")
    axes[1].set_xlabel("n_clusters (k)")
    axes[1].set_ylabel("Inertia")
    axes[1].set_title("KMeans Inertia（肘部法）")
plt.tight_layout()
plt.show()

## 5. 聚类质量与稳定性：如何比较

**不同聚类方法（同一 k）**  
- 看内部指标：**Silhouette** 越高越好、**Davies-Bouldin** 越低越好、**Calinski-Harabasz** 越高越好。  
- 可选：对每种方法跑一次 `run_patient_clustering_with_stability`，比较 **Bootstrap 平均 ARI** 和 95% CI，稳定性越高越可信。

**同一方法不同聚类数 k**  
- **Silhouette**：通常选使轮廓系数较高且随 k 变化出现“平台”或峰值的 k。  
- **肘部法（仅 KMeans）**：画 inertia 随 k 的曲线，选“拐点”对应的 k。  
- **稳定性**：对每个 k 跑 bootstrap ARI，选平均 ARI 高且 CI 较窄的 k。  
- 注意：不同 k 的 Silhouette 不能直接比大小（k 不同尺度不同），主要看趋势和拐点；跨 k 比较时更推荐用稳定性（bootstrap ARI）或轮廓系数随 k 的曲线形态。

下面是一段**可以直接写进实验报告/论文的方法学结果部分的综合结论**，逻辑包括：指标比较 → cluster size → 方法选择理由。

---

## 聚类方法选择的综合评估

为了确定最合适的患者亚型划分方法，我们比较了多种聚类算法在不同簇数（(K=2,3)）下的聚类性能。评估指标包括 **Silhouette score、Davies–Bouldin index 和 Calinski–Harabasz index**，分别用于衡量簇间分离度、簇内紧密度以及整体聚类结构质量。

首先，在不同簇数之间进行比较时发现，当 (K=2) 时，各方法的 **Silhouette score 整体高于 (K=3)**，同时 **Davies–Bouldin index 更低**，表明两类划分能够获得更清晰的簇间分离和更紧密的簇内结构。因此，从聚类质量指标来看，**(K=2) 相比 (K=3) 更为合适**。

在 (K=2) 条件下，我们进一步比较了不同聚类算法的结果。虽然 **average linkage hierarchical clustering** 获得了最高的 Silhouette score（0.434）和最低的 Davies–Bouldin index（0.704），但其产生的簇规模为 **37/492**，表现出明显的类别不平衡。这种极端不平衡的划分通常意味着算法识别出一个较小的异常群体，而非稳定且具有代表性的亚型结构。因此，该方法在实际分析中不具备良好的解释性和统计稳定性。

相比之下，**K-means clustering 在 (K=2) 时表现出更加合理的簇规模（285/244）**，两类样本数量基本均衡。同时，该方法获得了 **最高的 Calinski–Harabasz index（539）**，说明其在保持簇内紧密性的同时具有良好的簇间分离度。Spectral clustering 和 Ward hierarchical clustering 也得到相对合理的簇规模，但其综合指标略低于 K-means。

综合考虑 **聚类质量指标、簇规模平衡性以及方法稳定性**，本研究最终选择 **K-means clustering（(K=2)）作为主要的患者亚型划分方法**。该方法不仅在内部聚类指标上表现良好，而且能够产生规模合理且具有解释意义的患者分组，为后续的临床特征比较和脑网络分析提供可靠基础。

---

如果是**硕士论文/期刊论文更常见的简短版结论**，也可以写成一句话总结：

> 综合内部聚类指标和簇规模分布，K-means 在 (K=2) 时获得了最佳的综合表现（Silhouette=0.404，Calinski–Harabasz=539），且两类样本数量相对均衡（285/244）。因此，本研究选择 **K-means（(K=2)）** 作为最终的患者亚型划分方案。

---

如果你愿意，我也可以帮你把这段 **扩展成完整的 “Clustering evaluation” 小节（约300–400字）**，会更像 **NeuroImage / Brain 类论文的写法**。
